# Dataset 1 — Inspection Notebook

**Dataset:** Compact friction intervention testing  
**Raw folder:** `data_raw/1. data set, compact friction intervention testing/`  
**Purpose:** Structured inspection of all DS1 files before any cleaning is applied.  
**Companion notebook:** `02_dataset_1_cleaning.ipynb`  
**Audit report:** `outputs/reports/ds1_audit_report.md`  

This notebook follows the CLAUDE.md dataset-first workflow:  
inventory → documentation alignment → structural understanding → data quality audit.  

**No transformations are applied here.**

## 0. Setup

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Root is one level above notebooks/
ROOT = Path("..").resolve()
DS1_RAW = ROOT / "data_raw" / "1. data set, compact friction intervention testing"
REPORTS = ROOT / "outputs" / "reports"

print("Project root:", ROOT)
print("DS1 raw folder exists:", DS1_RAW.exists())

---
## 1. File Inventory

List every file in the DS1 folder and infer its role.

In [ ]:
all_files = sorted(DS1_RAW.rglob("*"))
for f in all_files:
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"{f.relative_to(DS1_RAW)}  [{size_kb:.0f} KB]")

In [ ]:
# File paths used throughout this notebook
BEH_CSV   = DS1_RAW / "Behavior data set" / "behavioral_data_set_1.csv"
BEH_XLSX  = DS1_RAW / "Behavior data set" / "behavioral_data_set_1.xlsx"
PRE_CSV   = DS1_RAW / "Survey data sets" / "Pre-survey" / "pre_survey_data_set_1.csv"
PRE_XLSX  = DS1_RAW / "Survey data sets" / "Pre-survey" / "pre_survey_data_set_1.xlsx"
POST_CSV  = DS1_RAW / "Survey data sets" / "Post-survey" / "post_survey_data_set_1.csv"
POST_XLSX = DS1_RAW / "Survey data sets" / "Post-survey" / "post_survey_data_set_1.xlsx"

print("All paths exist:", all(p.exists() for p in [BEH_CSV, BEH_XLSX, PRE_CSV, PRE_XLSX, POST_CSV, POST_XLSX]))

**File roles (confirmed):**

| File | Role |
|---|---|
| `behavioral_data_set_1.csv` | Event-level intervention data — primary behavioral substrate |
| `behavioral_data_set_1.xlsx` | Identical copy — not needed |
| `codebook_behavioral_data_set_1.pdf` | Variable documentation |
| `pre_survey_data_set_1.csv` | Pre-intervention survey |
| `pre_survey_data_set_1.xlsx` | Identical copy — not needed |
| `codebook_pre_survey_data_set_1.pdf` | Variable documentation |
| `post_survey_data_set_1.csv` | Post-intervention survey |
| `post_survey_data_set_1.xlsx` | Identical copy — not needed |
| `codebook_post_survey_data_set_1.pdf` | Variable documentation |
| `Project description, Compact default intervention study_.docx` | Study design document |

**Delimiter confirmed:** All three CSV files use semicolon (`;`) — not comma.

---
## 2. Behavioral Data
### 2.1 Load and basic structure

In [ ]:
# Confirm delimiter
with open(BEH_CSV) as f:
    print("First line:", repr(f.readline()))

In [ ]:
# IMPORTANT: sep=';' — file is semicolon-delimited, not comma
beh = pd.read_csv(BEH_CSV, sep=";")
print("Shape:", beh.shape)
print("Columns:", beh.columns.tolist())
print()
print(beh.dtypes)

In [ ]:
beh.head(5)

### 2.2 Timestamp format

**Confirmed:** `timestamp` is stored as `int64` Unix epoch (seconds).  
The codebook describes it as `year-month-day hour:minute:second` — this is a codebook error.  
Conversion: `pd.to_datetime(df['timestamp'], unit='s', utc=True)`  
No timezone column in the behavioral file — UTC assumed.

In [ ]:
sample_ts = beh["timestamp"].iloc[0]
print(f"Raw value:     {sample_ts}")
print(f"Converted UTC: {pd.to_datetime(sample_ts, unit='s', utc=True)}")
print()
ts_min = pd.to_datetime(beh['timestamp'].min(), unit='s', utc=True)
ts_max = pd.to_datetime(beh['timestamp'].max(), unit='s', utc=True)
span_days = (beh['timestamp'].max() - beh['timestamp'].min()) / 86400
print(f"Range: {ts_min}  →  {ts_max}")
print(f"Span:  {span_days:.1f} days")

### 2.3 Resolution values

**Confirmed 3 values (from codebook + CSV):**
- `openedApp` — user opened the app despite the intervention
- `dismissedAppOpening` — user dismissed the intervention
- `closedApp` — user closed the app after the intervention
- `NaN` — 468 rows with missing outcome (logging failure)

In [ ]:
print("Resolution value counts (incl. NaN):")
vc = beh["resolution"].value_counts(dropna=False)
pct = beh["resolution"].value_counts(dropna=False, normalize=True).round(4) * 100
summary = pd.DataFrame({"count": vc, "pct_%": pct})
print(summary.to_string())

### 2.4 Purpose — missingness

**Finding:** 71.3% of `purpose` values are NaN. Optional open-format comment field.  
Not reliable as a systematic moderator — treat as qualitative supplement only.

In [ ]:
n_null = beh["purpose"].isna().sum()
n_total = len(beh)
print(f"NaN purpose: {n_null:,} / {n_total:,}  ({n_null/n_total:.1%})")
print(f"Non-null:    {n_total-n_null:,} / {n_total:,}  ({(n_total-n_null)/n_total:.1%})")
print()
print("Sample non-null purpose values (random 15):")
print(beh["purpose"].dropna().sample(15, random_state=42).tolist())

### 2.5 secondsSinceOneSecSetup

**Codebook error:** The codebook lists this column as `secondsSinceStudySetup`.  
The actual column name in the CSV is `secondsSinceOneSecSetup`.  

**Meaning:** Seconds since the user first set up the OneSec app.  
Useful for longitudinal analysis (behavioral change over time since app setup).  

**18.9% NaN** — likely events before the study timer was initialized.

In [ ]:
col = "secondsSinceOneSecSetup"
n_null = beh[col].isna().sum()
print(f"NaN: {n_null:,} / {len(beh):,}  ({n_null/len(beh):.1%})")
print()
desc = beh[col].describe()
print(desc)
print()
print(f"Range: {beh[col].min()/86400:.1f}  to  {beh[col].max()/86400:.1f}  days since app setup")

### 2.6 Participants

In [ ]:
print(f"Unique participants (non-null UUID): {beh['participantUuid'].nunique()}")
print(f"NaN participantUuid rows: {beh['participantUuid'].isna().sum()}")
print()
ec = beh["participantUuid"].value_counts()
print("Events per participant:")
print(ec.describe().round(1))
print()
print("Top 5 most active:")
print(ec.head())
print()
print("Bottom 5 (fewest events):")
print(ec.tail())

In [ ]:
# The single NaN participantUuid row — inspect it
print("Row with NaN participantUuid:")
print(beh[beh["participantUuid"].isna()].to_string())

### 2.7 App distribution

In [ ]:
print(f"Unique apps: {beh['app'].nunique()}")
print()
print("Top 20 apps by event count:")
print(beh["app"].value_counts().head(20))

In [ ]:
# Typo check: 'whastapp' should be 'whatsapp'
whats = beh[beh["app"].str.lower().str.contains("what", na=False)]["app"].value_counts()
print("WhatsApp name variants (typo 'whastapp' is in the raw data):")
print(whats)

In [ ]:
# Resolution breakdown for top 5 apps
top5 = beh["app"].value_counts().head(5).index.tolist()
pivot = (
    beh[beh["app"].isin(top5)]
    .groupby(["app", "resolution"])
    .size()
    .unstack(fill_value=0)
)
pivot["open_rate"] = (pivot.get("openedApp", 0) / pivot.sum(axis=1)).round(3)
print(pivot)

### 2.8 Duplicate rows

**Finding:** 8,302 fully duplicate rows (all 6 columns identical) = ~2% of the data.  
Cannot be distinguished from genuine same-second events.  
**Cleaning rule:** drop all exact duplicates; document count.

In [ ]:
n_dupes = beh.duplicated().sum()
print(f"Fully duplicate rows: {n_dupes:,}  ({n_dupes/len(beh):.2%})")
print()
dupe_mask = beh.duplicated(keep=False)
dupe_rows = beh[dupe_mask]
print("Participants with most duplicate rows (top 10):")
print(dupe_rows["participantUuid"].value_counts().head(10))
print()
print("Apps most in duplicate rows (top 10):")
print(dupe_rows["app"].value_counts().head(10))
print()
first_dupe_id = dupe_rows["participantUuid"].value_counts().index[0]
print(f"Sample duplicate pair for participant {first_dupe_id[:8]}...:")
print(dupe_rows[dupe_rows["participantUuid"] == first_dupe_id].head(4).to_string())

### 2.9 NaN resolution rows

In [ ]:
nan_res = beh[beh["resolution"].isna()]
print(f"Rows with NaN resolution: {len(nan_res)}")
print()
print("Apps with NaN resolution (top 5):")
print(nan_res["app"].value_counts().head())
print()
print(f"Purpose fill rate for NaN-resolution rows: "
      f"{nan_res['purpose'].notna().sum()} / {len(nan_res)} non-null")
print()
print("Sample NaN-resolution rows (first 3):")
print(nan_res.head(3).to_string())

### 2.10 Full null summary

In [ ]:
null_beh = pd.DataFrame({
    "null_count": beh.isnull().sum(),
    "null_pct_%": (beh.isnull().sum() / len(beh) * 100).round(2)
})
print(null_beh.to_string())

---
## 3. Pre-Survey
### 3.1 Load and structure

In [ ]:
with open(PRE_CSV) as f:
    print("Delimiter check — first line:", repr(f.readline()[:120]))

In [ ]:
pre = pd.read_csv(PRE_CSV, sep=";")
print("Shape:", pre.shape)
print("Columns:", pre.columns.tolist())
print()
pre.head(3)

### 3.2 Consent

Participants with `I do NOT consent` must be excluded from all analyses.  
NaN consent = participant abandoned survey before the consent question.

In [ ]:
print("consent_general:")
print(pre["consent_general"].value_counts(dropna=False))

### 3.3 Likert scale storage format

**Finding:** All Likert columns store values as composite strings: `'5 - tendentially problematic'`  
The numeric prefix must be extracted with regex before any quantitative use.  
Extraction pattern: `r'^(\d+)'`

In [ ]:
likert_cols_pre = ["problematic_cons", "happy", "problematic_cons_general", "problematic_cons_compared"]
for col in likert_cols_pre:
    print(f"--- {col} ---")
    print(pre[col].value_counts(dropna=False).to_string())
    print()

### 3.4 Multi-select fields

**Finding:** `problems` and `goals` store multiple selections as comma-concatenated strings.  
Example: `'Excessive consumption,Feeling of no control,Automatic consumption'`  
Must be split into binary indicator columns before analysis.

In [ ]:
print("Top problems combinations (top 10):")
print(pre["problems"].value_counts(dropna=False).head(10))
print()
print("Top goals combinations (top 10):")
print(pre["goals"].value_counts(dropna=False).head(10))

### 3.5 Duplicate participantUuid

In [ ]:
n_u = pre["participantUuid"].nunique()
print(f"Unique IDs: {n_u}  |  Total rows: {len(pre)}  |  Duplicates: {len(pre)-n_u}")
print()
dupe_ids = pre[pre.duplicated("participantUuid", keep=False)]["participantUuid"].unique()
if len(dupe_ids) > 0:
    print(f"Example — participant with multiple pre-survey rows ({dupe_ids[0]}):")
    print(pre[pre["participantUuid"] == dupe_ids[0]].to_string())

### 3.6 Missingness summary

In [ ]:
null_pre = pd.DataFrame({
    "null_count": pre.isnull().sum(),
    "null_pct_%": (pre.isnull().sum() / len(pre) * 100).round(2)
})
print(null_pre.to_string())

---
## 4. Post-Survey
### 4.1 Load and structure

In [ ]:
post = pd.read_csv(POST_CSV, sep=";")
print("Shape:", post.shape)
print("Columns:", post.columns.tolist())
print()
post.head(3)

**Codebook note:** `consent_general` is present in the post-survey CSV but not documented in the codebook.  
Treated as equivalent to the pre-survey consent field.

In [ ]:
print("consent_general (post-survey, not in codebook):")
print(post["consent_general"].value_counts(dropna=False))

### 4.2 Likert scale storage

In [ ]:
likert_cols_post = ["problems_solved", "goals_reached", "problematic_cons",
                    "happy", "progress_satisfaction",
                    "problematic_cons_general", "problematic_cons_compared"]
for col in likert_cols_post:
    print(f"--- {col} ---")
    print(post[col].value_counts(dropna=False).to_string())
    print()

### 4.3 Duplicate participantUuid and missingness

In [ ]:
n_u_post = post["participantUuid"].nunique()
print(f"Unique IDs: {n_u_post}  |  Total rows: {len(post)}  |  Duplicates: {len(post)-n_u_post}")
print()
null_post = pd.DataFrame({
    "null_count": post.isnull().sum(),
    "null_pct_%": (post.isnull().sum() / len(post) * 100).round(2)
})
print(null_post.to_string())

---
## 5. CSV vs XLSX Identity Check

In [ ]:
beh_x  = pd.read_excel(BEH_XLSX)
pre_x  = pd.read_excel(PRE_XLSX)
post_x = pd.read_excel(POST_XLSX)

for name, csv_df, xlsx_df in [("Behavioral", beh, beh_x), ("Pre-survey", pre, pre_x), ("Post-survey", post, post_x)]:
    sm = csv_df.shape == xlsx_df.shape
    cm = csv_df.columns.tolist() == xlsx_df.columns.tolist()
    print(f"{name:<15}  shape match={sm}  column match={cm}")
print()
print("Verdict: all CSV/XLSX pairs are identical. Use CSV files only.")

---
## 6. Participant Overlap

In [ ]:
beh_ids  = set(beh["participantUuid"].dropna())
pre_ids  = set(pre["participantUuid"].dropna())
post_ids = set(post["participantUuid"].dropna())

groups = {
    "All three (full set)":   beh_ids & pre_ids & post_ids,
    "Beh + Pre only":         (beh_ids & pre_ids) - post_ids,
    "Beh + Post only":        (beh_ids & post_ids) - pre_ids,
    "Pre + Post only":        (pre_ids & post_ids) - beh_ids,
    "Behavioral only":        beh_ids - pre_ids - post_ids,
    "Pre only (no behavior)": pre_ids - beh_ids - post_ids,
    "Post only":              post_ids - beh_ids - pre_ids,
}
print("Participant overlap:")
for label, s in groups.items():
    print(f"  {label:<30}  {len(s):>5}")
print()
print(f"  Total behavioral:   {len(beh_ids):>5}")
print(f"  Total pre-survey:   {len(pre_ids):>5}")
print(f"  Total post-survey:  {len(post_ids):>5}")

**Key interpretations:**

| Group | N | Implication |
|---|---|---|
| All three | 169 | Fully analyzable for combined behavioral + psychosocial studies |
| Beh + Pre | 359 | Behavioral modeling with pre-intervention context; no post outcome |
| Behavioral only | 41 | Behavioral modeling only |
| **Pre only** | **575** | **Enrolled but no behavioral data — exclude from all behavioral modeling** |

The **619-participant behavioral file** is the primary substrate.  
The **169-participant fully-linked subset** is required for psychosocial moderator analyses.

---
## 7. Codebook Discrepancies

In [ ]:
discrepancies = [
    ("Medium", "Behavioral CSV",       "timestamp format: codebook says 'year-month-day'; actual is Unix epoch int64"),
    ("Low",    "Behavioral codebook",   "column listed as 'secondsSinceStudySetup'; actual CSV name is 'secondsSinceOneSecSetup'"),
    ("Low",    "Pre-survey codebook",   "'other_goals' description reads 'Outcome of intervention' — copy-paste error"),
    ("Low",    "Post-survey codebook",  "'progress_satisfaction' description reads 'Outcome of intervention' — copy-paste error"),
    ("Medium", "Post-survey codebook",  "'consent_general' present in CSV but absent from codebook"),
    ("Low",    "Behavioral data",        "app name 'whastapp' is a typo for 'whatsapp'"),
]
print(f"{'Sev':<8} {'File':<25} Issue")
print("-" * 80)
for sev, file, issue in discrepancies:
    print(f"[{sev:<6}] {file:<25} {issue}")

---
## 8. Inspection Summary

In [ ]:
print("=" * 65)
print("DS1 INSPECTION SUMMARY")
print("=" * 65)
print()
print("FILES & DELIMITER")
print(f"  Behavioral:   {beh.shape[0]:,} rows × {beh.shape[1]} cols  (sep=';')")
print(f"  Pre-survey:   {pre.shape[0]:,} rows × {pre.shape[1]} cols  (sep=';')")
print(f"  Post-survey:  {post.shape[0]:,} rows × {post.shape[1]} cols  (sep=';')")
print(f"  XLSX files:   identical to CSV — use CSV only")
print()
print("BEHAVIORAL KEY FACTS")
print(f"  Unique participants:           {beh['participantUuid'].nunique()}")
print(f"  Timestamp:                     Unix epoch int64 → convert with unit='s', utc=True")
print(f"  Resolution values:             openedApp / dismissedAppOpening / closedApp / NaN(468)")
print(f"  purpose NaN:                   {beh['purpose'].isna().mean():.1%}")
print(f"  secondsSinceOneSecSetup NaN:   {beh['secondsSinceOneSecSetup'].isna().mean():.1%}")
print(f"  Fully duplicate rows:          {beh.duplicated().sum():,}")
print(f"  Unique apps:                   {beh['app'].nunique()} (incl. 'whastapp' typo)")
print()
print("SURVEY KEY FACTS")
print(f"  Pre unique participants:       {pre['participantUuid'].nunique()} ({len(pre)-pre['participantUuid'].nunique()} duplicate rows)")
print(f"  Post unique participants:      {post['participantUuid'].nunique()} ({len(post)-post['participantUuid'].nunique()} duplicate rows)")
print(f"  Likert storage:                composite strings e.g. '5 - label' — extract int prefix")
print(f"  Multi-select storage:          comma-concatenated — split to binary indicators")
print()
print("PARTICIPANT OVERLAP")
beh_ids2 = set(beh['participantUuid'].dropna())
pre_ids2 = set(pre['participantUuid'].dropna())
post_ids2= set(post['participantUuid'].dropna())
print(f"  All three:                {len(beh_ids2 & pre_ids2 & post_ids2):>5}")
print(f"  Beh + Pre only:           {len((beh_ids2 & pre_ids2) - post_ids2):>5}")
print(f"  Behavioral only:          {len(beh_ids2 - pre_ids2 - post_ids2):>5}")
print(f"  Pre only (no behavior):   {len(pre_ids2 - beh_ids2 - post_ids2):>5}  ← exclude from behavioral modeling")
print()
print("Full audit report: outputs/reports/ds1_audit_report.md")